In [ ]:
%%capture
# Fix LD_LIBRARY_PATH for bitsandbytes CUDA detection
import os
os.environ['LD_LIBRARY_PATH'] = '/usr/local/cuda/lib64:/usr/local/nvidia/lib:/usr/local/nvidia/lib64'

# Uninstall existing bitsandbytes
!pip uninstall -y bitsandbytes

# Install PyTorch and other dependencies
!pip install -q torch torchvision torchaudio
!pip install -q transformers datasets accelerate
!pip install -q peft
!pip install -q sacrebleu
!pip install -q tensorboard
!pip install -q nltk
!pip install -q wandb
!pip install -q evaluate
!pip install -q jiwer  # For WER and CER
!pip install -q rouge-score  # For ROUGE

# Install bitsandbytes from source for CUDA 12.6 compatibility
!pip install bitsandbytes --no-cache-dir

# Verify
print("Installation complete!")

In [ ]:
import os
import gc

# Set CUDA paths BEFORE any CUDA imports
os.environ['LD_LIBRARY_PATH'] = '/usr/local/cuda/lib64:/usr/local/nvidia/lib:/usr/local/nvidia/lib64'
os.environ['CUDA_HOME'] = '/usr/local/cuda'
os.environ['BNB_CUDA_VERSION'] = '126'

import torch
import numpy as np
import pandas as pd
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, TaskType
from sacrebleu import BLEU
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

# Check CUDA availability
print(f"Number of GPUs: {torch.cuda.device_count()}")
print(f"GPU Names: {[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]}")
print(f"CUDA Version: {torch.version.cuda}")

Number of GPUs: 2
GPU Names: ['Tesla T4', 'Tesla T4']
CUDA Version: 12.6


[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [ ]:
# ============================================================================
# VERIFY BITSANDBYTES INSTALLATION
# ============================================================================

import os

# Set CUDA library paths before importing bitsandbytes
os.environ['LD_LIBRARY_PATH'] = '/usr/local/cuda/lib64:/usr/local/nvidia/lib:/usr/local/nvidia/lib64'
os.environ['CUDA_HOME'] = '/usr/local/cuda'
os.environ['BNB_CUDA_VERSION'] = '126'  # For CUDA 12.6

print("CUDA Environment Variables:")
print(f"  LD_LIBRARY_PATH: {os.environ.get('LD_LIBRARY_PATH', 'Not set')}")
print(f"  CUDA_HOME: {os.environ.get('CUDA_HOME', 'Not set')}")

import torch
print(f"\n✅ PyTorch CUDA available: {torch.cuda.is_available()}")
print(f"✅ PyTorch CUDA version: {torch.version.cuda}")
print(f"✅ Number of GPUs: {torch.cuda.device_count()}")

try:
    import bitsandbytes as bnb
    print(f"\n✅ bitsandbytes version: {bnb.__version__}")
    print("✅ bitsandbytes loaded successfully!")
except Exception as e:
    print(f"\n⚠️ bitsandbytes import failed: {e}")
    print("\nThis is OK - we'll use the fallback method without quantization.")

In [ ]:
# ============================================================================
# CONFIGURATION
# ============================================================================

MODEL_NAME = "facebook/nllb-200-3.3B"
OUTPUT_DIR = "/kaggle/working/nllb_qlora_checkpoint"
RESULTS_DIR = "/kaggle/working/results"

# Create directories
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# Data configuration
TRAIN_CSV_PATH = "/kaggle/input/datasets/farhanaadri/train-fin-vashantor/all_train_dataset.csv"  # 55k samples

# Dialect validation paths (per your original notebook)
DIALECT_VALIDATION_PATHS = {
    "Barishal": "/kaggle/input/datasets/farhanaadri/vashantor-dialect-to-english/Vashantor /Validation/Barishal  Validation Translation.csv",
    "Chittagong": "/kaggle/input/datasets/farhanaadri/vashantor-dialect-to-english/Vashantor /Validation/Chittagong Validation Translation.csv",
    "Mymensingh": "/kaggle/input/datasets/farhanaadri/vashantor-dialect-to-english/Vashantor /Validation/Mymensingh Validation Translation.csv",
    "Noakhali": "/kaggle/input/datasets/farhanaadri/vashantor-dialect-to-english/Vashantor /Validation/Noakhali Validation Translation.csv",
    "Sylhet": "/kaggle/input/datasets/farhanaadri/vashantor-dialect-to-english/Vashantor /Validation/Sylhet Validation Translation.csv"
}

DIALECT_BANGLA_COLS = {
    "Barishal": "barishal_bangla_speech ",
    "Chittagong": "chittagong_bangla_speech ",
    "Mymensingh": "mymensingh_bangla_speech ",
    "Noakhali": "noakhali_bangla_speech ",
    "Sylhet": "sylhet_bangla_speech "
}

# NLLB language codes
SRC_LANG = "ben_Beng"  # Bengali (Bangla)
TGT_LANG = "eng_Latn"  # English

# Training hyperparameters optimized for 2x T4 GPUs
BATCH_SIZE_PER_GPU = 2  # Reduced for 3.3B model
GRADIENT_ACCUMULATION_STEPS = 8  # Effective batch size: 2*2*8 = 32
NUM_EPOCHS = 1
LEARNING_RATE = 2e-4
MAX_SEQ_LENGTH = 256
EVAL_STEPS = 500
SAVE_STEPS = 500
WARMUP_STEPS = 500
WEIGHT_DECAY = 0.01

print("Configuration loaded successfully!")
print(f"Model: {MODEL_NAME}")
print(f"Source Language: {SRC_LANG}")
print(f"Target Language: {TGT_LANG}")

Configuration loaded successfully!


In [ ]:
# ============================================================================
# LOAD AND PREPARE DATA FOR NLLB
# ============================================================================

def prepare_training_data(csv_path):
    """
    Load CSV and prepare data for NLLB training.
    Expected columns: 'bangla_speech' (source) and 'english_speech' (target)
    """
    df = pd.read_csv(csv_path)
    
    # Fill missing values
    df["bangla_speech"] = df["bangla_speech"].fillna("").astype(str)
    df["english_speech"] = df["english_speech"].fillna("").astype(str)
    
    # Create source-target pairs
    sources = []
    targets = []
    dialects = []
    
    for idx, row in df.iterrows():
        source = row["bangla_speech"].strip()
        target = row["english_speech"].strip()
        
        # Skip empty pairs
        if not source or not target:
            continue
        
        sources.append(source)
        targets.append(target)
        
        # Extract dialect if available
        dialect = row.get("dialect", "unknown") if "dialect" in df.columns else "unknown"
        dialects.append(dialect)
    
    print(f"Prepared {len(sources)} training examples")
    return sources, targets, dialects

# Load training data
train_sources, train_targets, train_dialects = prepare_training_data(TRAIN_CSV_PATH)
print(f"Training samples: {len(train_sources)}")
print(f"Sample source: {train_sources[0][:100]}...")
print(f"Sample target: {train_targets[0][:100]}...")

Prepared 9374 training examples
Training samples: 9374
Sample: Translate from Bangla to English:
Bangla: লকট মোর প্রিয় ফল
English: Jamrul is my favorite fruit</s>...


In [ ]:
# Load NLLB tokenizer and prepare datasets
from transformers import AutoTokenizer, DataCollatorForSeq2Seq

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, src_lang=SRC_LANG, tgt_lang=TGT_LANG)

def tokenize_function(examples):
    """
    Tokenize source and target texts for NLLB seq2seq translation.
    """
    # Tokenize inputs
    model_inputs = tokenizer(
        examples["source"],
        max_length=MAX_SEQ_LENGTH,
        truncation=True,
        padding="max_length"
    )
    
    # Tokenize targets
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            examples["target"],
            max_length=MAX_SEQ_LENGTH,
            truncation=True,
            padding="max_length"
        )
    
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Create training dataset
train_dataset = Dataset.from_dict({
    "source": train_sources,
    "target": train_targets
})

train_dataset = train_dataset.map(
    tokenize_function,
    batched=True,
    batch_size=100,
    remove_columns=["source", "target"],
    num_proc=4
)

print(f"Tokenized training dataset: {train_dataset}")

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map (num_proc=4):   0%|          | 0/9374 [00:00<?, ? examples/s]

Tokenized training dataset: Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 9374
})


In [ ]:
# Load and prepare evaluation data with dialect tracking
def load_translation_dataset(csv_path, bangla_col):
    """
    Load a translation CSV and return a standardized DataFrame.
    """
    english_col = "english_speech"
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()

    # Keep only relevant columns
    df = df[[bangla_col.strip(), english_col.strip()]]

    # Rename columns to standard names
    df = df.rename(columns={
        bangla_col.strip(): "bangla_speech",
        english_col.strip(): "english_speech"
    })

    return df

# Load all dialect validation datasets
eval_by_dialect = {}

eval_sources = []
eval_targets = []

for dialect_name, csv_path in DIALECT_VALIDATION_PATHS.items():
    bangla_col = DIALECT_BANGLA_COLS[dialect_name]
    df = load_translation_dataset(csv_path, bangla_col)

    # Fill missing values
    df["bangla_speech"] = df["bangla_speech"].fillna("").astype(str)
    df["english_speech"] = df["english_speech"].fillna("").astype(str)

    sources = []
    targets = []

    for idx, row in df.iterrows():
        source = row["bangla_speech"].strip()
        target = row["english_speech"].strip()

        if source and target:  # Skip empty pairs
            sources.append(source)
            targets.append(target)
            eval_sources.append(source)
            eval_targets.append(target)

    eval_by_dialect[dialect_name] = {
        "sources": sources,
        "targets": targets
    }

    print(f"✓ {dialect_name}: {len(sources)} validation samples loaded")

print(f"\nTotal evaluation dialects: {len(eval_by_dialect)}")
print(f"Total combined eval samples: {len(eval_sources)}")

# Build combined eval dataset for Trainer
eval_dataset = Dataset.from_dict({
    "source": eval_sources,
    "target": eval_targets
})

eval_dataset = eval_dataset.map(
    tokenize_function,
    batched=True,
    batch_size=100,
    remove_columns=["source", "target"],
    num_proc=4
)

print(f"Tokenized eval dataset: {eval_dataset}")

✓ Barishal: 250 validation samples loaded
✓ Chittagong: 250 validation samples loaded
✓ Mymensingh: 250 validation samples loaded
✓ Noakhali: 250 validation samples loaded
✓ Sylhet: 250 validation samples loaded

Total evaluation dialects: 5
Total combined eval samples: 1250


Map (num_proc=4):   0%|          | 0/1250 [00:00<?, ? examples/s]

Tokenized eval dataset: Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 1250
})


In [ ]:
# ============================================================================
# CONFIGURE QLoRA + PEFT FOR NLLB
# ============================================================================

from transformers import AutoModelForSeq2SeqLM
from peft import prepare_model_for_kbit_training
import sys

# Try to import and verify bitsandbytes
try:
    import bitsandbytes as bnb
    print(f"✅ bitsandbytes imported successfully: {bnb.__version__}")
    bnb_available = True
except Exception as e:
    print(f"⚠️ bitsandbytes import failed: {e}")
    bnb_available = False

if not bnb_available:
    print("\n❌ FALLBACK: Loading model without quantization (will use more memory)")
    print("Loading NLLB model in bfloat16...")
    model = AutoModelForSeq2SeqLM.from_pretrained(
        MODEL_NAME,
        device_map="auto",
        torch_dtype=torch.bfloat16
    )
else:
    # BitsAndBytes configuration for 4-bit quantization
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16
    )

    print("Loading NLLB model with QLoRA quantization...")
    try:
        model = AutoModelForSeq2SeqLM.from_pretrained(
            MODEL_NAME,
            quantization_config=bnb_config,
            device_map="auto",
            torch_dtype=torch.bfloat16
        )
        print("✅ Model loaded with quantization")
    except Exception as e:
        print(f"⚠️ Quantization failed: {e}")
        print("Loading without quantization...")
        model = AutoModelForSeq2SeqLM.from_pretrained(
            MODEL_NAME,
            device_map="auto",
            torch_dtype=torch.bfloat16
        )

# CRITICAL: Disable cache before gradient checkpointing
model.config.use_cache = False

# Enable gradient checkpointing to save memory
model.gradient_checkpointing_enable()

# Prepare for k-bit training if quantized
if bnb_available:
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

print(f"Model loaded: {MODEL_NAME}")
print(f"Model dtype: {model.dtype}")
print(f"Gradient checkpointing enabled: {model.is_gradient_checkpointing}")

Loading model with QLoRA quantization...


config.json:   0%|          | 0.00/683 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


ImportError: Using `bitsandbytes` 4-bit quantization requires the latest version of bitsandbytes: `pip install -U bitsandbytes`

In [ ]:
# LoRA configuration for NLLB (Seq2Seq model) with full wildcarded target modules for all attention projections
lora_config = LoraConfig(
    r=16,  # LoRA rank
    lora_alpha=32,  # LoRA scaling
    target_modules=[
        # Encoder self-attention
        "encoder.layers.*.self_attn.q_proj",
        "encoder.layers.*.self_attn.k_proj",
        "encoder.layers.*.self_attn.v_proj",
        "encoder.layers.*.self_attn.out_proj",
        # Decoder self-attention
        "decoder.layers.*.self_attn.q_proj",
        "decoder.layers.*.self_attn.k_proj",
        "decoder.layers.*.self_attn.v_proj",
        "decoder.layers.*.self_attn.out_proj",
        # Decoder cross-attention (encoder_attn)
        "decoder.layers.*.encoder_attn.q_proj",
        "decoder.layers.*.encoder_attn.k_proj",
        "decoder.layers.*.encoder_attn.v_proj",
        "decoder.layers.*.encoder_attn.out_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM,  # Seq2Seq task
    inference_mode=False  # Training mode
)

# Apply LoRA to model (do not freeze manually; PEFT handles this)
model = get_peft_model(model, lora_config)

# Print trainable parameters info
model.print_trainable_parameters()

# Print dtype of a trainable parameter (if any)
trainable_params = [p for p in model.parameters() if p.requires_grad]
if trainable_params:
    print(f"Trainable parameters dtype: {trainable_params[0].dtype}")
else:
    print("No trainable parameters found! Check LoRA target modules.")

print("\nLoRA configured and applied for NLLB!")

In [ ]:
# ============================================================================
# DEBUG: PRINT ALL MODULE NAMES CONTAINING 'proj' TO FIND LoRA TARGETS
# ============================================================================
for name, module in model.named_modules():
    if 'proj' in name:
        print(name)

In [ ]:
# ============================================================================
# SETUP TRAINING ARGUMENTS FOR NLLB
# ============================================================================

from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    overwrite_output_dir=False,

    # Training hyperparameters
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE_PER_GPU,
    per_device_eval_batch_size=BATCH_SIZE_PER_GPU,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    weight_decay=WEIGHT_DECAY,

    # Multi-GPU distributed training
    ddp_find_unused_parameters=False,

    # Evaluation and saving
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=2,

    # Logging
    logging_dir=f"{OUTPUT_DIR}/logs",
    logging_steps=100,
    report_to=["tensorboard"],

    # Seq2Seq specific
    predict_with_generate=True,
    generation_max_length=MAX_SEQ_LENGTH,

    # Optimization
    optim="paged_adamw_32bit",
    fp16=False,
    bf16=True,

    # Distributed
    dataloader_num_workers=4,
    remove_unused_columns=False,

    # Other
    seed=42,
)

print("Training arguments configured for NLLB!")

In [ ]:
# Data collator for Seq2Seq (NLLB)
from transformers import Seq2SeqTrainer

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True
)

# Initialize Seq2Seq trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    tokenizer=tokenizer
)

print("Seq2Seq Trainer initialized for NLLB!")

Trainer initialized!


In [ ]:
# ============================================================================
# CUSTOM TRAINER TO FIX num_items_in_batch ERROR FOR NLLB
# ============================================================================
from transformers import Seq2SeqTrainer
class CustomSeq2SeqTrainer(Seq2SeqTrainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        # Remove num_items_in_batch for compatibility with M2M100/NLLB
        return super().compute_loss(model, inputs, return_outputs=return_outputs)

# Use the custom trainer instead of the default one
trainer = CustomSeq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    tokenizer=tokenizer
)

print("CustomSeq2SeqTrainer initialized for NLLB!")

In [ ]:
# ============================================================================
# TRAIN THE MODEL
# ============================================================================

print("Starting training...")
train_result = trainer.train()

print(f"\nTraining completed!")
print(f"Final loss: {train_result.training_loss}")

In [ ]:
# ============================================================================
# SAVE TRAINED MODEL
# ============================================================================

# Save the LoRA weights
model.save_pretrained(f"{OUTPUT_DIR}/final_lora")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/final_lora")

print(f"LoRA model saved to {OUTPUT_DIR}/final_lora")

In [ ]:
# ============================================================================
# IMPORTANT: SAVE MODEL IMMEDIATELY AFTER TRAINING
# ============================================================================
# This cell is a backup - run it if cell 13 fails for any reason

try:
    if 'model' in globals() and 'tokenizer' in globals():
        print("Backing up model and tokenizer...")
        model.save_pretrained(f"{OUTPUT_DIR}/final_lora")
        tokenizer.save_pretrained(f"{OUTPUT_DIR}/final_lora")
        print(f"✅ Backup save complete: {OUTPUT_DIR}/final_lora")
    else:
        print("⚠️ No model in memory to save. Run training cells 7-12 first.")
except Exception as e:
    print(f"❌ Save failed: {e}")

In [ ]:
# ============================================================================
# INFERENCE AND EVALUATION FOR NLLB
# ============================================================================

import os
import glob
from peft import PeftModel

# Strategy: Try multiple sources in order of preference
model_for_inference = None
base_model = None

# Option 1: Load from final_lora if it exists
final_lora_path = f"{OUTPUT_DIR}/final_lora"
if os.path.exists(f"{final_lora_path}/adapter_config.json"):
    print(f"✅ Loading model from: {final_lora_path}")
    base_model = AutoModelForSeq2SeqLM.from_pretrained(
        MODEL_NAME,
        device_map="auto",
        torch_dtype=torch.bfloat16
    )
    model_for_inference = PeftModel.from_pretrained(base_model, final_lora_path)
    print("Model loaded successfully!")

# Option 2: Load from latest checkpoint if final_lora doesn't exist
elif os.path.exists(OUTPUT_DIR):
    checkpoints = sorted(glob.glob(f"{OUTPUT_DIR}/checkpoint-*"))
    if checkpoints:
        latest_checkpoint = checkpoints[-1]
        print(f"✅ Loading from latest checkpoint: {latest_checkpoint}")
        base_model = AutoModelForSeq2SeqLM.from_pretrained(
            MODEL_NAME,
            device_map="auto",
            torch_dtype=torch.bfloat16
        )
        model_for_inference = PeftModel.from_pretrained(base_model, latest_checkpoint)
        print("Model loaded successfully!")

# Option 3: Use model from current session
if model_for_inference is None:
    if 'model' in globals():
        print("✅ Using model from current training session")
        model_for_inference = model
    else:
        raise RuntimeError(
            "❌ No model found!\n"
            "Please run training cells first."
        )

print("\n✓ NLLB model ready for inference!")

In [ ]:
def generate_translation(source_text, model, tokenizer, max_length=128):
    """
    Generate translation using NLLB model.
    """
    try:
        # Tokenize with forced source language
        inputs = tokenizer(
            source_text,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_SEQ_LENGTH
        )
        
        # Move to model device
        device = next(model.parameters()).device
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        # Generate with forced target language
        with torch.no_grad():
            generated_tokens = model.generate(
                **inputs,
                forced_bos_token_id=tokenizer.lang_code_to_id[TGT_LANG],
                max_length=max_length,
                num_beams=4,
                early_stopping=True
            )
        
        # Decode the generated translation
        translation = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)[0]
        return translation.strip()
        
    except Exception as e:
        print(f"⚠️ Translation error: {e}")
        return "[ERROR]"

# Test with a few examples
print("\n=== Sample NLLB Translations ===")
for dialect, data in eval_by_dialect.items():
    if len(data["sources"]) > 0:
        source = data["sources"][0]
        reference = data["targets"][0]
        prediction = generate_translation(source, model_for_inference, tokenizer)
        
        print(f"\nDialect: {dialect}")
        print(f"Source:      {source[:80]}...")
        print(f"Reference:   {reference[:80]}...")
        print(f"Prediction:  {prediction[:80]}...")

In [ ]:
# ============================================================================
# COMPREHENSIVE EVALUATION WITH MULTIPLE METRICS
# ============================================================================

import sacrebleu
from jiwer import wer, cer
from rouge_score import rouge_scorer
import evaluate

# Load METEOR metric
meteor_metric = evaluate.load('meteor')

def normalize_texts(texts):
    """Normalize whitespace and strip texts."""
    normalized = []
    for t in texts:
        t = t.strip()
        t = " ".join(t.split())  # normalize whitespace
        normalized.append(t)
    return normalized

def compute_bleu_score(predictions, references):
    """Compute corpus-level BLEU score using SacreBLEU."""
    predictions = normalize_texts(predictions)
    references = normalize_texts(references)
    bleu_score = sacrebleu.corpus_bleu(predictions, [references], tokenize="13a")
    return bleu_score

def compute_wer_cer(predictions, references):
    """Compute Word Error Rate and Character Error Rate."""
    predictions = normalize_texts(predictions)
    references = normalize_texts(references)
    
    wer_score = wer(references, predictions)
    cer_score = cer(references, predictions)
    
    return wer_score, cer_score

def compute_meteor(predictions, references):
    """Compute METEOR score."""
    predictions = normalize_texts(predictions)
    references = normalize_texts(references)
    
    meteor_score = meteor_metric.compute(predictions=predictions, references=references)
    return meteor_score['meteor']

def compute_rouge(predictions, references):
    """Compute ROUGE scores (ROUGE-1, ROUGE-2, ROUGE-L)."""
    predictions = normalize_texts(predictions)
    references = normalize_texts(references)
    
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    
    rouge1_f = []
    rouge2_f = []
    rougeL_f = []
    
    for pred, ref in zip(predictions, references):
        scores = scorer.score(ref, pred)
        rouge1_f.append(scores['rouge1'].fmeasure)
        rouge2_f.append(scores['rouge2'].fmeasure)
        rougeL_f.append(scores['rougeL'].fmeasure)
    
    return {
        'rouge1': sum(rouge1_f) / len(rouge1_f),
        'rouge2': sum(rouge2_f) / len(rouge2_f),
        'rougeL': sum(rougeL_f) / len(rougeL_f)
    }

print("\n" + "="*80)
print("COMPREHENSIVE EVALUATION BY DIALECT")
print("="*80)

# Dictionary to store results
bleu_results = {}

for dialect, data in eval_by_dialect.items():
    print(f"\n📊 Evaluating dialect: {dialect}")
    print(f"   Samples: {len(data['sources'])}")
    
    # Generate predictions for all samples in this dialect
    predictions = []
    for i, source in enumerate(data["sources"]):
        if i % max(1, len(data['sources']) // 5) == 0:  # Progress every 20%
            print(f"   Processing: {i}/{len(data['sources'])}...")
        
        try:
            prediction = generate_translation(source, model_for_inference, tokenizer)
            predictions.append(prediction)
        except Exception as e:
            print(f"   Warning: Failed on sample {i}: {str(e)}")
            predictions.append("")  # Add empty prediction on failure
    
    # Compute all metrics
    print(f"   Computing metrics...")
    
    # BLEU
    bleu_score = compute_bleu_score(predictions, data["targets"])
    
    # WER and CER
    wer_score, cer_score = compute_wer_cer(predictions, data["targets"])
    
    # METEOR
    meteor_score = compute_meteor(predictions, data["targets"])
    
    # ROUGE
    rouge_scores = compute_rouge(predictions, data["targets"])
    
    bleu_results[dialect] = {
        "bleu": bleu_score.score,
        "precisions": bleu_score.precisions,
        "bp": bleu_score.bp,
        "ratio": bleu_score.sys_len / bleu_score.ref_len if bleu_score.ref_len > 0 else 0,
        "translation_length": bleu_score.sys_len,
        "reference_length": bleu_score.ref_len,
        "wer": wer_score * 100,  # Convert to percentage
        "cer": cer_score * 100,  # Convert to percentage
        "meteor": meteor_score * 100,  # Convert to percentage
        "rouge1": rouge_scores['rouge1'] * 100,
        "rouge2": rouge_scores['rouge2'] * 100,
        "rougeL": rouge_scores['rougeL'] * 100
    }
    
    print(f"   ✅ BLEU Score: {bleu_score.score:.2f}")
    print(f"   ✅ WER: {bleu_results[dialect]['wer']:.2f}%")
    print(f"   ✅ CER: {bleu_results[dialect]['cer']:.2f}%")
    print(f"   ✅ METEOR: {bleu_results[dialect]['meteor']:.2f}%")
    print(f"   ✅ ROUGE-1: {bleu_results[dialect]['rouge1']:.2f}%")
    print(f"   ✅ ROUGE-2: {bleu_results[dialect]['rouge2']:.2f}%")
    print(f"   ✅ ROUGE-L: {bleu_results[dialect]['rougeL']:.2f}%")
    print(f"   Precisions (1-4gram): {[f'{p:.4f}' for p in bleu_score.precisions]}")
    print(f"   Brevity Penalty: {bleu_score.bp:.4f}")

print("\n" + "="*80)

In [ ]:
# ============================================================================
# SAVE AND VISUALIZE COMPREHENSIVE RESULTS
# ============================================================================

# Save results to CSV
results_df = pd.DataFrame([
    {
        "dialect": dialect,
        "bleu_score": scores["bleu"],
        "wer": scores["wer"],
        "cer": scores["cer"],
        "meteor": scores["meteor"],
        "rouge1": scores["rouge1"],
        "rouge2": scores["rouge2"],
        "rougeL": scores["rougeL"],
        "precision_1gram": scores["precisions"][0],
        "precision_2gram": scores["precisions"][1],
        "precision_3gram": scores["precisions"][2],
        "precision_4gram": scores["precisions"][3],
        "brevity_penalty": scores["bp"],
        "length_ratio": scores["ratio"],
        "translation_length": scores["translation_length"],
        "reference_length": scores["reference_length"]
    }
    for dialect, scores in bleu_results.items()
])

results_csv_path = f"{RESULTS_DIR}/comprehensive_evaluation_results.csv"
results_df.to_csv(results_csv_path, index=False)
print(f"\nResults saved to {results_csv_path}")

# Display results
print("\n" + "="*80)
print("FINAL RESULTS SUMMARY")
print("="*80)
print(results_df[["dialect", "bleu_score", "wer", "cer", "meteor", "rouge1", "rouge2", "rougeL"]].to_string(index=False))

# Overall averages
print(f"\n📈 Average Metrics Across All Dialects:")
print(f"   BLEU:    {results_df['bleu_score'].mean():.2f}")
print(f"   WER:     {results_df['wer'].mean():.2f}%")
print(f"   CER:     {results_df['cer'].mean():.2f}%")
print(f"   METEOR:  {results_df['meteor'].mean():.2f}%")
print(f"   ROUGE-1: {results_df['rouge1'].mean():.2f}%")
print(f"   ROUGE-2: {results_df['rouge2'].mean():.2f}%")
print(f"   ROUGE-L: {results_df['rougeL'].mean():.2f}%")

In [ ]:
# ============================================================================
# VISUALIZE COMPREHENSIVE RESULTS
# ============================================================================

import matplotlib.pyplot as plt

# Plot comprehensive metrics
fig, axes = plt.subplots(3, 2, figsize=(16, 14))
fig.suptitle('NLLB Fine-tuning: Comprehensive Evaluation by Dialect', fontsize=16, fontweight='bold')

# Plot 1: BLEU Scores
ax = axes[0, 0]
ax.bar(results_df["dialect"], results_df["bleu_score"], color='steelblue')
ax.set_ylabel('BLEU Score')
ax.set_title('BLEU Scores by Dialect')
ax.tick_params(axis='x', rotation=45)
ax.grid(axis='y', alpha=0.3)

# Plot 2: WER and CER
ax = axes[0, 1]
x = range(len(results_df))
width = 0.35
ax.bar([i - width/2 for i in x], results_df["wer"], width, label='WER', color='coral')
ax.bar([i + width/2 for i in x], results_df["cer"], width, label='CER', color='lightcoral')
ax.set_ylabel('Error Rate (%)')
ax.set_title('Word Error Rate (WER) and Character Error Rate (CER)')
ax.set_xticks(x)
ax.set_xticklabels(results_df["dialect"], rotation=45)
ax.legend()
ax.grid(axis='y', alpha=0.3)

# Plot 3: METEOR Score
ax = axes[1, 0]
ax.bar(results_df["dialect"], results_df["meteor"], color='mediumseagreen')
ax.set_ylabel('METEOR Score (%)')
ax.set_title('METEOR Scores by Dialect')
ax.tick_params(axis='x', rotation=45)
ax.grid(axis='y', alpha=0.3)

# Plot 4: ROUGE Scores
ax = axes[1, 1]
ax.plot(results_df["dialect"], results_df["rouge1"], marker='o', label='ROUGE-1', linewidth=2)
ax.plot(results_df["dialect"], results_df["rouge2"], marker='s', label='ROUGE-2', linewidth=2)
ax.plot(results_df["dialect"], results_df["rougeL"], marker='^', label='ROUGE-L', linewidth=2)
ax.set_ylabel('ROUGE Score (%)')
ax.set_title('ROUGE Scores by Dialect')
ax.legend()
ax.tick_params(axis='x', rotation=45)
ax.grid(alpha=0.3)

# Plot 5: N-gram Precisions
ax = axes[2, 0]
precision_cols = ["precision_1gram", "precision_2gram", "precision_3gram", "precision_4gram"]
for col in precision_cols:
    ax.plot(results_df["dialect"], results_df[col], marker='o', label=col.replace('_', ' ').title())
ax.set_ylabel('Precision')
ax.set_title('N-gram Precisions by Dialect')
ax.legend()
ax.tick_params(axis='x', rotation=45)
ax.grid(alpha=0.3)

# Plot 6: Overall Metrics Comparison
ax = axes[2, 1]
metrics_summary = {
    'BLEU': results_df['bleu_score'].mean(),
    'METEOR': results_df['meteor'].mean(),
    'ROUGE-1': results_df['rouge1'].mean(),
    'ROUGE-L': results_df['rougeL'].mean()
}
colors = ['steelblue', 'mediumseagreen', 'orange', 'purple']
ax.bar(metrics_summary.keys(), metrics_summary.values(), color=colors)
ax.set_ylabel('Score')
ax.set_title('Average Scores Across All Dialects')
ax.tick_params(axis='x', rotation=0)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/comprehensive_evaluation_results.png", dpi=300, bbox_inches='tight')
print(f"\nVisualization saved to {RESULTS_DIR}/comprehensive_evaluation_results.png")
plt.show()

In [ ]:
# ============================================================================
# INFERENCE EXAMPLES FOR EACH DIALECT
# ============================================================================

inference_examples = []

for dialect, data in eval_by_dialect.items():
    if len(data["sources"]) > 0:
        # Sample 3 examples from each dialect
        sample_indices = np.random.choice(len(data["sources"]), min(3, len(data["sources"])), replace=False)
        
        for idx in sample_indices:
            source = data["sources"][idx]
            reference = data["targets"][idx]
            prediction = generate_translation(source, model_for_inference, tokenizer)
            
            inference_examples.append({
                "dialect": dialect,
                "source": source,
                "reference": reference,
                "prediction": prediction
            })

# Save inference examples
inference_df = pd.DataFrame(inference_examples)
inference_csv_path = f"{RESULTS_DIR}/inference_examples.csv"
inference_df.to_csv(inference_csv_path, index=False)
print(f"Inference examples saved to {inference_csv_path}")

# Display a few examples
print("\n" + "="*80)
print("INFERENCE EXAMPLES")
print("="*80)
for i, example in enumerate(inference_examples[:5]):
    print(f"\n\nExample {i+1} ({example['dialect']})")
    print(f"Source:      {example['source'][:100]}...")
    print(f"Reference:   {example['reference'][:100]}...")
    print(f"Prediction:  {example['prediction'][:100]}...")

In [ ]:
# ============================================================================
# CLEAN UP AND SUMMARY
# ============================================================================

print("\n" + "="*80)
print("TRAINING COMPLETED SUCCESSFULLY")
print("="*80)

print(f"\n✅ Model checkpoint: {OUTPUT_DIR}/final_lora")
print(f"✅ Results saved to: {RESULTS_DIR}")
print(f"✅ CSV results: {results_csv_path}")
print(f"✅ Inference examples: {inference_csv_path}")
print(f"✅ Visualizations: {RESULTS_DIR}/comprehensive_evaluation_results.png")

# Clean up GPU memory
gc.collect()
torch.cuda.empty_cache()

print(f"\n🎯 Training Configuration:")
print(f"   - Model: {MODEL_NAME}")
print(f"   - Training samples: {len(train_sources)}")
print(f"   - Evaluation samples: {sum(len(d['sources']) for d in eval_by_dialect.values())}")
print(f"   - Epochs: {NUM_EPOCHS}")
print(f"   - Batch size (effective): {BATCH_SIZE_PER_GPU * 2 * GRADIENT_ACCUMULATION_STEPS}")
print(f"   - LoRA rank: 16, Alpha: 32")
print(f"   - Quantization: 4-bit (NF4)")
print(f"   - Hardware: 2x T4 GPUs")

print(f"\n📊 Final Average Metrics:")
print(f"   - BLEU:    {results_df['bleu_score'].mean():.2f}")
print(f"   - WER:     {results_df['wer'].mean():.2f}%")
print(f"   - CER:     {results_df['cer'].mean():.2f}%")
print(f"   - METEOR:  {results_df['meteor'].mean():.2f}%")
print(f"   - ROUGE-1: {results_df['rouge1'].mean():.2f}%")
print(f"   - ROUGE-2: {results_df['rouge2'].mean():.2f}%")
print(f"   - ROUGE-L: {results_df['rougeL'].mean():.2f}%")